# Part 5: Random Forest Deployment — AWS SageMaker


In [ ]:
# STEP 1: Downgrade sklearn to match AWS container version
# After this cell finishes, go to Kernel → Restart Kernel, then run the next cell
import sys
!{sys.executable} -m pip install scikit-learn==1.2.3 "sagemaker<3.0"
print("\n✅ Done! Now click Kernel → Restart Kernel, then run Step 2 cell below.")

In [ ]:
# STEP 2: Run this AFTER restarting the kernel
import sklearn
print(f"sklearn version: {sklearn.__version__}")
assert sklearn.__version__.startswith('1.2'), 'ERROR: Please restart kernel first!'

import pandas as pd, numpy as np, sagemaker, joblib, tarfile, json, time, os
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sagemaker.sklearn.model import SKLearnModel
from sagemaker.serverless import ServerlessInferenceConfig

# --- Feature Engineering from raw CSV ---
print('Loading dataset...')
df = pd.read_csv('homes(1).csv')
df = df[df['JustValue'] > 0].copy()

df['PropertyAge']      = 2024 - df['YearBuilt']
df['BathToBedRatio']   = df['TotalBathrooms'] / (df['TotalBedrooms'] + 1)
df['HasExtraFeatures'] = (df['TotalExtraFeaturesValue'] > 0).astype(int)
df['LuxuryIndicator']  = (df['JustValue'] > df['JustValue'].quantile(0.85)).astype(int)
df['MedianValueByZip'] = df.groupby('SiteZip')['JustValue'].transform('median')

le_type = LabelEncoder()
le_city = LabelEncoder()
df['PropertyType_enc'] = le_type.fit_transform(df['PropertyType'].fillna('Unknown'))
df['SiteCity_enc']     = le_city.fit_transform(df['SiteCity'].fillna('Unknown'))

FEATURES = [
    'TotalHeatedAreaSqFt', 'TotalBedrooms', 'TotalBathrooms',
    'PropertyAge', 'Acreage', 'HasExtraFeatures', 'LuxuryIndicator',
    'TotalNumBuildings', 'MedianValueByZip', 'BathToBedRatio',
    'PropertyType_enc', 'SiteCity_enc'
]
df = df.dropna(subset=FEATURES + ['JustValue'])
X, y = df[FEATURES], df['JustValue']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Training on {len(X_train):,} properties | Testing on {len(X_test):,} properties')

# --- Train Compact RandomForest with sklearn 1.2.3 ---
print('\nTraining RandomForest...')
model = RandomForestRegressor(
    n_estimators=25, max_depth=10,
    min_samples_leaf=5, random_state=42, n_jobs=-1
)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
r2   = r2_score(y_test, y_pred)
mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f'\nModel Performance on {len(X_test):,} test properties:')
print(f'  R² Score : {r2:.4f}')
print(f'  MAE      : ${mae:,.0f}')
print(f'  RMSE     : ${rmse:,.0f}')

joblib.dump(model, 'rf_model.joblib')
size_mb = os.path.getsize('rf_model.joblib') / (1024*1024)
print(f'  Model size: {size_mb:.1f} MB')

In [ ]:
# STEP 3: Write inference script, package and deploy
with open('infer_rf.py', 'w') as f:
    f.write('''import joblib, os, json, pandas as pd

FEATURES = ["TotalHeatedAreaSqFt","TotalBedrooms","TotalBathrooms","PropertyAge",
            "Acreage","HasExtraFeatures","LuxuryIndicator","TotalNumBuildings",
            "MedianValueByZip","BathToBedRatio","PropertyType_enc","SiteCity_enc"]

def model_fn(model_dir):
    return joblib.load(os.path.join(model_dir, "rf_model.joblib"))

def input_fn(request_body, content_type):
    df = pd.DataFrame(json.loads(request_body)["instances"])
    for col in FEATURES:
        if col not in df.columns: df[col] = 0
    return df[FEATURES]

def predict_fn(input_data, model):
    return model.predict(input_data)

def output_fn(prediction, content_type):
    return json.dumps({"JustValue_Predictions": prediction.tolist()})
''')
print('inference script written!')

# Package
with tarfile.open('model_rf.tar.gz', 'w:gz') as tar:
    tar.add('rf_model.joblib')

# Upload
session       = sagemaker.Session()
role          = sagemaker.get_execution_role()
model_s3_path = session.upload_data('model_rf.tar.gz')
endpoint_name = f'real-estate-rf-{int(time.time())}'
print(f'Deploying RandomForest endpoint: {endpoint_name} ...')

# Deploy
predictor = SKLearnModel(
    model_data=model_s3_path, role=role,
    entry_point='infer_rf.py',
    framework_version='1.2-1', py_version='py3'
).deploy(
    serverless_inference_config=ServerlessInferenceConfig(
        memory_size_in_mb=3072, max_concurrency=1
    ),
    endpoint_name=endpoint_name
)
print(f'\n✅ RandomForest DEPLOYED: {predictor.endpoint_name}')

In [ ]:
# STEP 4: Invoke the live endpoint
import boto3
client = boto3.client('sagemaker-runtime', region_name=session.boto_region_name)

sample = {
    "instances": [{
        "TotalHeatedAreaSqFt": 2150, "TotalBedrooms": 3,
        "TotalBathrooms": 2.0,        "PropertyAge": 12,
        "Acreage": 0.25,              "HasExtraFeatures": 1,
        "LuxuryIndicator": 0,         "TotalNumBuildings": 1,
        "MedianValueByZip": 280000,   "BathToBedRatio": 0.67,
        "PropertyType_enc": 0,        "SiteCity_enc": 1
    }]
}

response = client.invoke_endpoint(
    EndpointName=predictor.endpoint_name,
    ContentType='application/json',
    Body=json.dumps(sample)
)
result = json.loads(response['Body'].read().decode('utf-8'))
print(f'\n🏠 Live AWS RandomForest Prediction:')
print(f'   Predicted JustValue: ${result["JustValue_Predictions"][0]:,.0f}')